# Minimal WEkEO example: download a subset with the HDA API and plot it

This notebook follows WEkEO's official guide *How to download WEkEO data with the HDA API in JupyterHub* and adds a plotting step at the end.

**Before you start**
- Run it on the WEkEO JupyterHub with the **Earth Observation Tools** server and the **Python 3 (ipykernel)** kernel. The `hda` package is already installed there.
- You need a WEkEO account. Register at https://wekeo.copernicus.eu/register.

**Getting your own query.** Open the Expert Data Viewer at https://wekeo.copernicus.eu/data, add a dataset and set the area, time and variables in the subsetting form. Then click **Show API request** and copy the JSON into the query cell below. The example query here is the one used in WEkEO's own guide. It asks for sea water salinity from the Copernicus Marine Global Ocean Physics Analysis and Forecast product, off the French Mediterranean coast.

## 1. Import `hda` and connect with your WEkEO credentials

In [ ]:
from hda import Client, Configuration
import getpass

# Same username and password you use on wekeo.copernicus.eu.
# getpass keeps the password out of the saved notebook.
USERNAME = input("WEkEO username: ")
PASSWORD = getpass.getpass("WEkEO password: ")

conf = Configuration(user=USERNAME, password=PASSWORD)
hda_client = Client(config=conf)

## 2. Define the query (paste from *Show API request*)

In [ ]:
query = {
  "dataset_id": "EO:MO:DAT:GLOBAL_ANALYSISFORECAST_PHY_001_024:cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m_202406",
  "minimum_depth": 0.494024992,
  "maximum_depth": 100,
  "startdate": "2026-06-01T00:00:00.000Z",
  "enddate": "2026-06-07T23:59:59.999Z",
  "bbox": [
    4.3595919567022055,
    41.893317734758384,
    5.561835484362211,
    42.60122926047297
  ],
  "variables": [
    "so"
  ],
  "itemsPerPage": 200,
  "startIndex": 0
}

## 3. Search and download

In [ ]:
# Search for files matching the query
matches = hda_client.search(query)

# List the results
print(matches)

In [ ]:
# Download all results into a directory
download_dir = "data"
matches.download(download_dir=download_dir)

## 4. Open and plot the data

In [ ]:
from pathlib import Path
import zipfile
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some datasets are delivered zipped. Unpack them if so.
for z in Path(download_dir).glob("*.zip"):
    zipfile.ZipFile(z).extractall(download_dir)

nc_files = sorted(Path(download_dir).glob("*.nc"))
print(nc_files)

ds = xr.open_dataset(nc_files[0])
ds

In [ ]:
# First day, shallowest depth level
so = ds["so"].isel(time=0, depth=0)

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
so.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="viridis",
        cbar_kwargs={"label": "Salinity [PSU]"})
ax.coastlines(resolution="10m")
ax.add_feature(cfeature.LAND, facecolor="lightgrey")
ax.gridlines(draw_labels=True, linestyle=":")
ax.set_title(f"Sea water salinity, {str(so.time.values)[:10]}, {float(so.depth):.1f} m")
plt.show()

## Notes

- To get the files onto your own computer, right-click them in the file browser and choose **Download**. For many files, zip the folder first with `shutil.make_archive("my_data", "zip", ".", download_dir)`.
- If `print(matches)` shows 0 results, check the dates and bounding box against the dataset's coverage, and rebuild the query in the Viewer.
- Your home directory holds up to 20 GB.